# Create_dataset — Stage A (PyBullet → `Trajectory.npz`)

Notebook này đi theo **cùng luồng** với hướng dẫn step-by-step:

1. **Sanity**: chạy `unittest` cho `create_dataset_module`.
2. **Config A (smoke)**: `n_scenes=10`, `rollouts_per_scene=2`, `frames_per_rollout=200` → thư mục `data/stage_a_smoke/`.
3. **(Tuỳ chọn)** `python -m create_dataset_module.verify` — kiểm tra thêm RiskDataset + collate (+ FullPipeline nếu có CUDA/checkpoint).
4. **(Tuỳ chọn)** Config B (dataset lớn) — chỉnh tham số trong cell cuối.

---

### Máy local (Cursor / VS Code / Jupyter)

- Mở notebook **từ thư mục repo** `Pipeline/` (cùng cấp với `create_dataset_module/`).
- Chọn **kernel Python = `.venv`** của project (`…\\Pipeline\\.venv\\Scripts\\python.exe`) để dùng đúng thư viện đã cài.
- Cell đầu sẽ in `sys.executable` — kiểm tra đường dẫn có chứa `.venv`.

### Google Colab

- Zip toàn bộ source từ máy (xem `create_dataset_module/README.md` — cần `PointPillars_module/`, `create_dataset_module/`, `urdf/`, `pybullet_navigation.py`, `run_generate_small.py`, `run_datagen_preset.py`, `rgb_preview_to_png.py`, `rgb_preview_layout.py`, …).
- Upload zip trong cell **Colab setup**, giải nén vào `/content/Pipeline`, cài `pybullet` + `matplotlib`.
- Không bắt buộc GPU để **sinh dataset**; GPU chỉ hữu ích nếu sau đó chạy PointPillars / training.

In [13]:
"""Resolve repo root + Python dùng cho mọi bước sau (kernel / .venv / Colab)."""
from __future__ import annotations

import os
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False


def find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for _ in range(8):
        if (cur / "create_dataset_module").is_dir() and (cur / "PointPillars_module").is_dir():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    raise RuntimeError(
        "Không tìm thấy root repo (cần có create_dataset_module/ và PointPillars_module/). "
        f"cwd={Path.cwd()!s}. Hãy mở notebook trong Pipeline/ hoặc chạy cell Colab setup trước."
    )


if not IN_COLAB:
    ROOT = find_repo_root(Path.cwd())
    os.chdir(ROOT)
else:
    # Sẽ được set lại sau khi giải nén zip (cell Colab setup)
    try:
        ROOT = find_repo_root(Path.cwd())
        os.chdir(ROOT)
    except RuntimeError:
        ROOT = Path("/content/Pipeline").resolve()

for p in (str(ROOT), str(ROOT / "PointPillars_module")):
    if p not in sys.path:
        sys.path.insert(0, p)


def pipeline_python_exe(root: Path) -> Path:
    """Python cho subprocess (unittest, script): ưu tiên `.venv` — có torch/pybullet đúng bản."""
    if sys.platform == "win32":
        cand = root / ".venv" / "Scripts" / "python.exe"
    else:
        cand = root / ".venv" / "bin" / "python"
    return cand if cand.is_file() else Path(sys.executable)


PIPELINE_PY = pipeline_python_exe(ROOT)

print("IN_COLAB    :", IN_COLAB)
print("ROOT        :", ROOT)
print("cwd         :", Path.cwd())
print("kernel (sys.executable) :", sys.executable)
print("PIPELINE_PY (subprocess) :", PIPELINE_PY)
if PIPELINE_PY.resolve() != Path(sys.executable).resolve():
    print("  → Kernel khác .venv nhưng unittest/script vẫn dùng PIPELINE_PY trong .venv.")

IN_COLAB    : False
ROOT        : E:\RobotDog_Project\Pipeline
cwd         : E:\RobotDog_Project\Pipeline
kernel (sys.executable) : c:\Users\quyen\AppData\Local\Programs\Python\Python310\python.exe
PIPELINE_PY (subprocess) : E:\RobotDog_Project\Pipeline\.venv\Scripts\python.exe
  → Kernel khác .venv nhưng unittest/script vẫn dùng PIPELINE_PY trong .venv.


## Colab — upload zip & cài dependency (bỏ qua nếu chạy local)

Chỉ chạy cell này trên Colab. Zip nên được tạo từ **thư mục gốc repo** (trong zip có `create_dataset_module/`, `PointPillars_module/`, …).

In [ ]:
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

try:
    import google.colab
    from google.colab import files
except ImportError:
    print("Không phải Colab — bỏ qua cell này.")
else:
    uploaded = files.upload()
    assert uploaded, "Upload file .zip chứa source Pipeline."
    zip_name = next(iter(uploaded.keys()))
    work_root = Path("/content/Pipeline")
    if work_root.exists():
        shutil.rmtree(work_root)
    work_root.mkdir(parents=True)
    with zipfile.ZipFile(zip_name) as z:
        z.extractall(work_root)
    os.chdir(work_root)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pybullet", "matplotlib"])
    for p in (str(work_root), str(work_root / "PointPillars_module")):
        if p not in sys.path:
            sys.path.insert(0, p)
    print("Working dir:", os.getcwd())
    print("Top-level  :", sorted(os.listdir("."))[:30], "...")
    print("sys.path[0:3]:", sys.path[:3])

### (Colab) Làm mới `ROOT` sau khi upload

Nếu đã chạy cell upload zip ở trên, có thể **chạy lại cell đầu tiên** (resolve `ROOT`) hoặc chạy cell code ngay dưới — tùy một trong hai.

In [ ]:
# Sau cell Colab upload: chạy lại resolve ROOT (hoặc chạy lại cell đầu tiên)
from pathlib import Path
import os
import sys

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False


def find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for _ in range(8):
        if (cur / "create_dataset_module").is_dir() and (cur / "PointPillars_module").is_dir():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    raise RuntimeError("Chưa tìm thấy repo — chạy cell upload zip trước.")


ROOT = find_repo_root(Path.cwd())
os.chdir(ROOT)
for p in (str(ROOT), str(ROOT / "PointPillars_module")):
    if p not in sys.path:
        sys.path.insert(0, p)


def pipeline_python_exe(root):
    from pathlib import Path as P

    if sys.platform == "win32":
        cand = root / ".venv" / "Scripts" / "python.exe"
    else:
        cand = root / ".venv" / "bin" / "python"
    return cand if cand.is_file() else P(sys.executable)


PIPELINE_PY = pipeline_python_exe(ROOT)
print("ROOT:", ROOT)
print("PIPELINE_PY:", PIPELINE_PY)

## Bước 1 — Unit test `create_dataset_module`

Chạy **cell đầu tiên** trước (để có `PIPELINE_PY`). Test gọi Python trong `.venv` (`PIPELINE_PY`), không nhất thiết trùng kernel Jupyter — tránh lỗi `No module named 'torch'` khi kernel là Python hệ thống.

In [14]:
import subprocess
import sys
from pathlib import Path

# Cần cell đầu (ROOT, PIPELINE_PY). Không dùng sys.executable — kernel có thể là Python hệ thống (thiếu torch).
ROOT = Path.cwd().resolve()
r = subprocess.run(
    [
        str(PIPELINE_PY),
        "-m",
        "unittest",
        "discover",
        "-s",
        "create_dataset_module/tests",
        "-t",
        ".",
        "-v",
    ],
    cwd=str(ROOT),
    capture_output=True,
    text=True,
)
print(r.stdout, end="")
if r.stderr:
    print(r.stderr, end="", file=sys.stderr)
if r.returncode != 0:
    raise RuntimeError(
        f"unittest thất bại (exit {r.returncode}). "
        "Thường do kernel không phải .venv: cell này đã gọi PIPELINE_PY từ cell đầu. "
        "Chạy lại cell đầu tiên, hoặc kiểm tra lỗi ImportError ở trên."
    )
print("\n[OK] unittest create_dataset_module")

W/S/A/D = Control robot | R = Reset | Q = Quit
W/S/A/D = Control robot | R = Reset | Q = Quit
W/S/A/D = Control robot | R = Reset | Q = Quit
W/S/A/D = Control robot | R = Reset | Q = Quit
W/S/A/D = Control robot | R = Reset | Q = Quit

 DataGenerator summary  (out_dir = C:\Users\quyen\AppData\Local\Temp\tmpjx09d7dq)
   rollouts written     : 1
   frames total         : 251
   early-terminated     : 1 (100.0%)
   positive ratio 0.5s  : 3.98%
   positive ratio 1.0s  : 7.97%
   positive ratio 2.0s  : 15.94%
   policy random      : 0  (0.0%)
   policy scripted    : 0  (0.0%)
   policy adversarial : 1  (100.0%)
W/S/A/D = Control robot | R = Reset | Q = Quit

 DataGenerator summary  (out_dir = C:\Users\quyen\AppData\Local\Temp\tmp_sed3b_o)
   rollouts written     : 2
   frames total         : 30
   early-terminated     : 0 (0.0%)
   positive ratio 0.5s  : 0.00%
   positive ratio 1.0s  : 0.00%
   positive ratio 2.0s  : 0.00%
   policy random      : 1  (50.0%)
   policy scripted    : 0  (0.0%)

pybullet build time: Apr 18 2026 08:46:11
test_medium_trajectory_preserves_all_fields (create_dataset_module.tests.test_contracts_roundtrip.TestRoundtrip) ... ok
test_jitter_magnitude_is_small (create_dataset_module.tests.test_domain_rand.TestCameraJitter) ... ok
test_jitter_preserves_orthogonality (create_dataset_module.tests.test_domain_rand.TestCameraJitter) ... ok
test_zero_jitter_is_identity (create_dataset_module.tests.test_domain_rand.TestCameraJitter) ... ok
test_nonzero_std_perturbs_and_stays_in_range (create_dataset_module.tests.test_domain_rand.TestDepthNoise) ... ok
test_zero_std_is_identity (create_dataset_module.tests.test_domain_rand.TestDepthNoise) ... ok
test_nonzero_prob_creates_zeros (create_dataset_module.tests.test_domain_rand.TestPixelDropout) ... ok
test_zero_prob_is_identity (create_dataset_module.tests.test_domain_rand.TestPixelDropout) ... ok
test_camera_spec_matches_indoor_defaults (create_dataset_module.tests.test_env_wrapper.TestDatasetEnv) ... ok
test_dept

## Bước 2 — Config A (smoke): gọi `run_generate_small.py`

Script nằm ở root repo; tương đương `python run_generate_small.py` trong terminal (cùng `.venv`).

Output: `data/stage_a_smoke/` + bảng summary (`positive ratio`, v.v.).

In [15]:
import subprocess
import sys
from pathlib import Path

# Repo root: ưu tiên ROOT từ cell 1 (đã os.chdir)
try:
    base = ROOT
except NameError:
    base = Path.cwd().resolve()


def _pipeline_py(repo: Path) -> Path:
    """Trùng logic cell 1: ưu tiên repo/.venv — có torch."""
    if sys.platform == "win32":
        cand = repo / ".venv" / "Scripts" / "python.exe"
    else:
        cand = repo / ".venv" / "bin" / "python"
    return cand if cand.is_file() else Path(sys.executable)


py = _pipeline_py(base)
script = base / "run_generate_small.py"
if not script.is_file():
    raise FileNotFoundError(f"Thiếu {script} — chạy cell đầu (cd vào repo) hoặc copy file vào zip Colab.")

print("run_generate_small dùng Python:", py)
if py.resolve() == Path(sys.executable).resolve():
    print(
        "⚠ Không thấy .venv trong repo — đang dùng kernel. "
        "Nếu lỗi No module named 'torch': tạo venv tại repo hoặc chọn kernel Pipeline\\.venv."
    )

r = subprocess.run(
    [str(py), str(script)],
    cwd=str(base),
    capture_output=True,
    text=True,
)
print(r.stdout, end="")
if r.stderr:
    print(r.stderr, end="", file=sys.stderr)
if r.returncode != 0:
    raise RuntimeError(
        f"run_generate_small.py exit {r.returncode}. Xem stderr phía trên. "
        "Gợi ý: thiếu torch (sai Python), hoặc matplotlib backend inline khi chạy subprocess từ Jupyter "
        "(đã xử lý trong pybullet_navigation.py — pull code mới nhất)."
    )
print("\n[OK] run_generate_small.py")

run_generate_small dùng Python: E:\RobotDog_Project\Pipeline\.venv\Scripts\python.exe
[config A] starting small smoke generation ...
  target frames (pre-term) = 4000
  out_dir = data/stage_a_smoke
W/S/A/D = Control robot | R = Reset | Q = Quit
W/S/A/D = Control robot | R = Reset | Q = Quit
W/S/A/D = Control robot | R = Reset | Q = Quit
W/S/A/D = Control robot | R = Reset | Q = Quit
W/S/A/D = Control robot | R = Reset | Q = Quit
W/S/A/D = Control robot | R = Reset | Q = Quit
W/S/A/D = Control robot | R = Reset | Q = Quit
W/S/A/D = Control robot | R = Reset | Q = Quit
W/S/A/D = Control robot | R = Reset | Q = Quit
W/S/A/D = Control robot | R = Reset | Q = Quit

 DataGenerator summary  (out_dir = data\stage_a_smoke)
   rollouts written     : 20
   frames total         : 3910
   early-terminated     : 3 (15.0%)
   positive ratio 0.5s  : 0.77%
   positive ratio 1.0s  : 1.53%
   positive ratio 2.0s  : 3.07%
   policy random      : 8  (40.0%)
   policy scripted    : 6  (30.0%)
   policy adve

pybullet build time: Apr 18 2026 08:46:11


### (Tuỳ chọn) Cùng Config A → `data/stage_a_smoke_nb` (subprocess, **không cần torch trong kernel**)

Cell dưới gọi `run_datagen_preset.py smoke_nb` bằng **`.venv\Scripts\python.exe`** — giống Bước 2, kernel Jupyter có thể là Python hệ thống.

Muốn chỉnh tham số: sửa preset `smoke_nb` trong `run_datagen_preset.py` hoặc dùng `run_generate_small.py` / terminal.

In [18]:
import os
import subprocess
import sys
from pathlib import Path

try:
    _repo = ROOT
except NameError:
    _repo = Path.cwd().resolve()
ROOT = Path(_repo).resolve()
os.chdir(ROOT)


def _pipeline_py(repo: Path) -> Path:
    if sys.platform == "win32":
        cand = repo / ".venv" / "Scripts" / "python.exe"
    else:
        cand = repo / ".venv" / "bin" / "python"
    return cand if cand.is_file() else Path(sys.executable)


py = _pipeline_py(ROOT)
script = ROOT / "run_datagen_preset.py"
if not script.is_file():
    raise FileNotFoundError(f"Thiếu {script}")

print("preset smoke_nb — Python:", py)
r = subprocess.run(
    [str(py), str(script), "smoke_nb"],
    cwd=str(ROOT),
    capture_output=True,
    text=True,
)
print(r.stdout, end="")
if r.stderr:
    print(r.stderr, end="", file=sys.stderr)
if r.returncode != 0:
    raise RuntimeError(f"run_datagen_preset.py exit {r.returncode}")
print("\n[OK] data/stage_a_smoke_nb")

preset smoke_nb — Python: E:\RobotDog_Project\Pipeline\.venv\Scripts\python.exe
W/S/A/D = Control robot | R = Reset | Q = Quit
W/S/A/D = Control robot | R = Reset | Q = Quit
W/S/A/D = Control robot | R = Reset | Q = Quit
W/S/A/D = Control robot | R = Reset | Q = Quit
W/S/A/D = Control robot | R = Reset | Q = Quit
W/S/A/D = Control robot | R = Reset | Q = Quit
W/S/A/D = Control robot | R = Reset | Q = Quit
W/S/A/D = Control robot | R = Reset | Q = Quit
W/S/A/D = Control robot | R = Reset | Q = Quit
W/S/A/D = Control robot | R = Reset | Q = Quit

 DataGenerator summary  (out_dir = E:\RobotDog_Project\Pipeline\data\stage_a_smoke_nb)
   rollouts written     : 20
   frames total         : 3910
   early-terminated     : 3 (15.0%)
   positive ratio 0.5s  : 0.77%
   positive ratio 1.0s  : 1.53%
   positive ratio 2.0s  : 3.07%
   policy random      : 8  (40.0%)
   policy scripted    : 6  (30.0%)
   policy adversarial : 6  (30.0%)
   WARN  positive(1s) < 5% -> consider raising policy_adversarial

pybullet build time: Apr 18 2026 08:46:11


## Bước 3 (tuỳ chọn) — Smoke end-to-end: `create_dataset_module.verify`

Tạo dataset tạm, load `RiskDataset`, collate; bước PointPillars/`FullPipeline` có thể bị skip nếu không có CUDA/checkpoint.

In [ ]:
import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
r = subprocess.run(
    [
        str(PIPELINE_PY),
        "-m",
        "create_dataset_module.verify",
        "--tmp_dir",
        str(ROOT / "_verify_tmp"),
        "--keep",
    ],
    cwd=str(ROOT),
)
if r.returncode != 0:
    raise RuntimeError(f"verify thất bại (exit {r.returncode})")
print("\n[OK] verify")

## Bước 4 (tuỳ chọn) — Dataset lớn (overnight / Colab)

Ví dụ: 200 scene × 4 rollout × 400 frame. Chỉ chạy khi đã hài lòng **positive ratio** từ bước 2 (thường mong ~10–15% cho risk 1s).

Cell code dưới gọi `run_datagen_preset.py full` qua subprocess (**không cần** kernel có `torch`).

In [17]:
import os
import subprocess
import sys
from pathlib import Path

try:
    _repo = ROOT
except NameError:
    _repo = Path.cwd().resolve()
ROOT = Path(_repo).resolve()
os.chdir(ROOT)


def _pipeline_py(repo: Path) -> Path:
    if sys.platform == "win32":
        cand = repo / ".venv" / "Scripts" / "python.exe"
    else:
        cand = repo / ".venv" / "bin" / "python"
    return cand if cand.is_file() else Path(sys.executable)


py = _pipeline_py(ROOT)
script = ROOT / "run_datagen_preset.py"
if not script.is_file():
    raise FileNotFoundError(f"Thiếu {script}")

print("preset full — Python:", py)
r = subprocess.run(
    [str(py), str(script), "full"],
    cwd=str(ROOT),
    capture_output=True,
    text=True,
)
print(r.stdout, end="")
if r.stderr:
    print(r.stderr, end="", file=sys.stderr)
if r.returncode != 0:
    raise RuntimeError(f"run_datagen_preset.py exit {r.returncode}")
print("\n[OK] data/stage_a_full (lâu — nhiều giờ)")

ModuleNotFoundError: No module named 'torch'. Chọn kernel Jupyter: E:\RobotDog_Project\Pipeline\.venv\Scripts\python.exe — hoặc sinh data bằng script/subprocess thay vì import trong notebook.

---

## Legit cho bài toán train (không chỉ smoke test)

Smoke (`run_generate_small`, `smoke_nb`) chỉ để **test pipeline**. Trước khi train thật, nên kiểm:

1. **Scale đủ lớn** — ví dụ hướng dẫn: ~200 scene × 4 rollout × 400 frame (hoặc tương đương tổng frame); tránh dừng ở vài chục rollout.
2. **Positive ratio / mix policy trong vùng chấp nhận** — xem summary cuối `DataGenerator.run()`: thường mong **positive(1s) ~10–15%** (không quá thấp để model không học rủi ro, không quá cao để không lệch toàn “sắp đâm”). Điều chỉnh `policy_random_p` / `policy_scripted_p` / `policy_adversarial_p` cho tới khi WARN biến mất hoặc ratio nằm vùng mong muốn.
3. **(Tuỳ chọn) RGB trên vài rollout** — chạy các cell **「Spot-check RGB」** ở cuối notebook (preset `rgb_spotcheck` + hiển thị frame), hoặc tự đặt `save_rgb=True` trong `DataGenConfig` cho vài rollout ngắn; sau đó tắt lại để dataset train không phình dung lượng.

Khi train: dùng **scene-stratified split** (`scene_stratified_split`) để train/val/test không trùng layout scene.

---

## Ghi chú nhanh

- **Positive ratio 1s** trong summary: tỷ lệ frame được gán nguy cơ va chạm trong ~1s tới. Nếu quá thấp → tăng `policy_adversarial_p`; quá cao → giảm.
- **Colab**: nhớ zip kèm `urdf/diff_drive_2wheel.urdf` và `pybullet_navigation.py`, không chỉ `create_dataset_module/`.
- Notebook cũ (demo r2d2 + `build_dataset` thủ công) đã được thay bằng pipeline `DataGenerator` + `DataGenConfig` hiện tại.

### (Tuỳ chọn) Spot-check RGB — kiểm sim bằng ảnh

Chạy **cell 1** (có `ROOT`) trước. Cell dưới gọi preset `rgb_spotcheck` (1 rollout ngắn, `save_rgb=True`) → `data/stage_a_rgb_spotcheck/`. Cell tiếp theo: nếu kernel có `matplotlib` thì vẽ trong notebook; nếu không, gọi `rgb_preview_to_png.py` bằng `.venv` và ghi `rgb_preview.png` — mở file trong Explorer (kernel **không cần** `torch`).

Sau khi xem xong, dataset train thật vẫn nên để `save_rgb=False` (mặc định).

In [26]:
import os
import subprocess
import sys
from pathlib import Path

try:
    _repo = ROOT
except NameError:
    _repo = Path.cwd().resolve()
ROOT = Path(_repo).resolve()
os.chdir(ROOT)


def _pipeline_py(repo: Path) -> Path:
    if sys.platform == "win32":
        cand = repo / ".venv" / "Scripts" / "python.exe"
    else:
        cand = repo / ".venv" / "bin" / "python"
    return cand if cand.is_file() else Path(sys.executable)


py = _pipeline_py(ROOT)
script = ROOT / "run_datagen_preset.py"
if not script.is_file():
    raise FileNotFoundError(f"Thiếu {script}")

print("preset rgb_spotcheck — Python:", py)
r = subprocess.run(
    [str(py), str(script), "rgb_spotcheck"],
    cwd=str(ROOT),
    capture_output=True,
    text=True,
)
print(r.stdout, end="")
if r.stderr:
    print(r.stderr, end="", file=sys.stderr)
if r.returncode != 0:
    raise RuntimeError(f"rgb_spotcheck exit {r.returncode}")
print("\n[OK] data/stage_a_rgb_spotcheck — chạy cell tiếp theo để xem ảnh")

preset rgb_spotcheck — Python: E:\RobotDog_Project\Pipeline\.venv\Scripts\python.exe
W/S/A/D = Control robot | R = Reset | Q = Quit

 DataGenerator summary  (out_dir = E:\RobotDog_Project\Pipeline\data\stage_a_rgb_spotcheck)
   rollouts written     : 1
   frames total         : 120
   early-terminated     : 0 (0.0%)
   positive ratio 0.5s  : 0.00%
   positive ratio 1.0s  : 0.00%
   positive ratio 2.0s  : 0.00%
   policy random      : 0  (0.0%)
   policy scripted    : 1  (100.0%)
   policy adversarial : 0  (0.0%)
   policy stationary  : 0  (0.0%)
   WARN  positive(1s) < 5% -> consider raising policy_adversarial_p
last_stats: {'written': 1, 'total_frames': 120, 'positive_ratio_05s': 0.0, 'positive_ratio_1s': 0.0, 'positive_ratio_2s': 0.0, 'policy_counts': {'random': 0, 'scripted': 1, 'adversarial': 0, 'stationary': 0}, 'terminated_early': 0}

[OK] data/stage_a_rgb_spotcheck — chạy cell tiếp theo để xem ảnh


pybullet build time: Apr 18 2026 08:46:11


In [27]:
# Hiển thị RGB: numpy trong kernel; matplotlib tùy chọn — nếu kernel thiếu matplotlib,
# gọi rgb_preview_to_png.py bằng .venv (luôn có matplotlib từ pybullet).
from pathlib import Path
import subprocess
import sys

import numpy as np

try:
    _repo = ROOT
except NameError:
    _repo = Path.cwd().resolve()
ROOT = Path(_repo).resolve()


def _pipeline_py(repo: Path) -> Path:
    if sys.platform == "win32":
        cand = repo / ".venv" / "Scripts" / "python.exe"
    else:
        cand = repo / ".venv" / "bin" / "python"
    return cand if cand.is_file() else Path(sys.executable)


out = ROOT / "data/stage_a_rgb_spotcheck"
files = sorted(out.glob("*.npz"))
if not files:
    raise FileNotFoundError(f"Chưa có .npz trong {out} — chạy cell rgb_spotcheck phía trên.")

z = np.load(files[0])
rgb = z["rgb"]
print(files[0].name, "rgb shape:", rgb.shape, "dtype:", rgb.dtype)

if rgb.size == 0 or rgb.ndim < 3:
    raise RuntimeError("RGB rỗng — kiểm tra save_rgb=True và preset đã chạy xong.")

try:
    import matplotlib.pyplot as plt

    from rgb_preview_layout import grid_rows_cols, sample_frame_indices

    T = rgb.shape[0]
    idxs = sample_frame_indices(T, 30)
    n = len(idxs)
    rows, cols = grid_rows_cols(n)
    fig, axs = plt.subplots(rows, cols, figsize=(cols * 2.15, rows * 2.05))
    axs = np.atleast_1d(axs).ravel()
    for k in range(rows * cols):
        ax = axs[k]
        if k < n:
            ax.imshow(rgb[idxs[k]])
            ax.set_title(f"t={idxs[k]}", fontsize=8)
        ax.axis("off")
    fig.suptitle("RGB montage (time →)", fontsize=10, y=1.02)
    plt.tight_layout()
    plt.show()
except ModuleNotFoundError:
    py = _pipeline_py(ROOT)
    helper = ROOT / "rgb_preview_to_png.py"
    if not helper.is_file():
        raise FileNotFoundError(f"Thiếu {helper}")
    png = out / "rgb_preview.png"
    r = subprocess.run(
        [str(py), str(helper), str(files[0]), str(png)],
        cwd=str(ROOT),
        capture_output=True,
        text=True,
    )
    print(r.stdout, end="")
    if r.stderr:
        print(r.stderr, end="", file=sys.stderr)
    if r.returncode != 0:
        raise RuntimeError("rgb_preview_to_png.py thất bại")
    print(f"Kernel không có matplotlib — đã ghi {png}. Mở file ảnh trong Explorer.")

s0000_r00.npz rgb shape: (120, 120, 160, 3) dtype: uint8
Wrote E:\RobotDog_Project\Pipeline\data\stage_a_rgb_spotcheck\rgb_preview.png
Kernel không có matplotlib — đã ghi E:\RobotDog_Project\Pipeline\data\stage_a_rgb_spotcheck\rgb_preview.png. Mở file ảnh trong Explorer.
